In [42]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [43]:
# Load dataset
df = pd.read_csv('datasets/german_credit_data.csv')
print(df.shape)
print(df.head())

(1000, 11)
   Unnamed: 0  Age     Sex  Job Housing Saving accounts Checking account  \
0           0   67    male    2     own             NaN           little   
1           1   22  female    2     own          little         moderate   
2           2   49    male    1     own          little              NaN   
3           3   45    male    2    free          little           little   
4           4   53    male    2    free          little           little   

   Credit amount  Duration              Purpose  Risk  
0           1169         6             radio/TV  good  
1           5951        48             radio/TV   bad  
2           2096        12            education  good  
3           7882        42  furniture/equipment  good  
4           4870        24                  car   bad  


In [44]:
# Drop index column if present
if 'Unnamed: 0' in df.columns:
    df.drop(columns=['Unnamed: 0'], inplace=True)

In [45]:
df.columns = df.columns.str.lower().str.replace(' ', '_')
df.columns

Index(['age', 'sex', 'job', 'housing', 'saving_accounts', 'checking_account',
       'credit_amount', 'duration', 'purpose', 'risk'],
      dtype='str')

In [46]:
# Drop missing values in account records
df = df.dropna().reset_index(drop=True)

In [47]:
# Target check: Good (1 / Low Risk) vs Bad (0 / High Risk)
print(df['risk'].value_counts(normalize=True) * 100)

risk
good    55.747126
bad     44.252874
Name: proportion, dtype: float64


In [48]:
import joblib
from sklearn.preprocessing import LabelEncoder

In [49]:
features = ['age', 'sex', 'job', 'housing', 'saving_accounts', 'checking_account', 'credit_amount', 'duration']
target = 'risk'

In [50]:
df_model = df[features + [target]].copy()

In [51]:
# Categorical column encoding
cat_cols = ['sex', 'housing', 'saving_accounts', 'checking_account']
encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col])
    encoders[col] = le
    joblib.dump(le, f'{col}_encoder.pickle')

In [52]:
# Target encoding (1 = Good, 0 = Bad)
le_target = LabelEncoder()
df_model[target] = le_target.fit_transform(df_model[target])
joblib.dump(le_target, 'target_encoder.pickle')

['target_encoder.pickle']

In [53]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier

In [54]:
# Train/Test Split
X = df_model.drop(columns=['risk'])
y = df_model['risk']

In [55]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=1
)

In [56]:
# Helper function for grid search
def train_model(model, param_grid):
    grid = GridSearchCV(model, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train, y_train)
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    return best_model, acc, grid.best_params_

In [57]:
# Evaluate ExtraTreesClassifier (Best Performing Model ~66.6% accuracy)
et = ExtraTreesClassifier(random_state=1, class_weight='balanced', n_jobs=-1)
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

In [58]:
best_et, acc_et, params_et = train_model(et, param_grid)
print(f"Extra Trees Accuracy: {acc_et:.4f}")
print("Best Params:", params_et)

Extra Trees Accuracy: 0.6476
Best Params: {'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}


In [59]:
# Save trained model
joblib.dump(best_et, 'extra_trees_credit_model.pickle')

['extra_trees_credit_model.pickle']